# TCL4 E2E CPU vs CUDA compare (spin-boson)

This notebook contains two parts:

1) A **full spin-boson TCL simulation** via `taco.tcl.simulate(...)` (CPU and optionally CUDA).
2) An **E2E CPU vs CUDA compare** that mirrors the C++ tool `tcl4_e2e_cuda_compare`.

Tip: the native extension (`taco/_taco*.pyd`) must match the Python version of your selected Jupyter kernel.

Notes:
- Use `precision="fp64"` for FP64 kernels.
- Use `precision="fp32"` for FP32 kernels; for perf-only benchmarking, set `check=False`.


In [1]:
from __future__ import annotations

import os
import sys
from pathlib import Path

# Make the in-repo package importable when running from a repo checkout.
# This tries a few common layouts (repo root, notebook folder, or home-dir launches).
# Tip: you can also set `TACO_REPO` to the repo root.
cwd = Path.cwd().resolve()
home = Path.home().resolve()
env_root = os.environ.get("TACO_REPO") or os.environ.get("TACO_ROOT")

python_dir = None
candidates = [cwd, *cwd.parents, home, *home.parents]
for base in candidates:
    # base is the repo root
    if (base / "python" / "taco").is_dir():
        python_dir = base / "python"
        break
    # base is the `python/` folder
    if (base / "taco" / "__init__.py").is_file():
        python_dir = base
        break
    # base is a parent of the repo root (common if Jupyter started from your home dir)
    for repo_root in (
        base / "taco",
        base / "repos" / "taco",
        base / "src" / "taco",
        base / "projects" / "taco",
    ):
        if (repo_root / "python" / "taco").is_dir():
            python_dir = repo_root / "python"
            break
    if python_dir is not None:
        break

if python_dir is None and env_root:
    repo_root = Path(env_root).expanduser().resolve()
    if (repo_root / "python" / "taco").is_dir():
        python_dir = repo_root / "python"

if python_dir is None:
    raise RuntimeError(
        f"Could not locate the repo's `python/` directory from cwd={cwd}. "
        "Start Jupyter from the repo root, or set TACO_REPO to the repo root."
    )

sys.path.insert(0, str(python_dir))
print("Using python_dir:", python_dir)

import taco

print("taco.version() =", taco.version())
print("taco.build_info() =", taco.build_info())
print("taco.cuda.is_available() =", taco.cuda.is_available())
print("taco.cuda.device_count() =", taco.cuda.device_count())


Using python_dir: C:\Users\59405\taco\python
taco.version() = taco_tcl-0.1
taco.build_info() = {'cuda_enabled': True, 'openmp_enabled': True, 'compiler': {'id': 'msvc', 'version': '1940'}, 'cuda_runtime_version': 12050}
taco.cuda.is_available() = True
taco.cuda.device_count() = 1


## Spin-boson model (TLS) + bath + simulation settings

This section runs a **full TCL simulation** from Python via `taco.tcl.simulate(...)`.

**Model**
- Hamiltonian (lab basis): `H = 0.5*eps*sigma_z + 0.5*Delta*sigma_x`
- Coupling operator (one channel): `A = 0.5*sigma_z` (this is the operator used by the backend)

**Bath**
- Tabulated spectral density: provide arrays `omega` and `J(omega)` (float64)
- Example below: `J(ω) = alpha * ω * exp(-ω/ωc)`
- `omega` should be increasing; `bcf_end_time` should cover the simulation time window you care about

**Simulation**
- `dt`, `n_steps`, `save_stride`, `order` (0/2/4)
- `save_stride`: save every k-th step to control output size
- `bcf_end_time`: memory horizon used when computing the bath correlation function
- Time integration uses dense RK4 internally (CPU or CUDA depending on `device`)


In [ ]:
import numpy as np

def sigma_x() -> np.ndarray:
    return np.array([[0.0, 1.0], [1.0, 0.0]], dtype=np.complex128)

def sigma_z() -> np.ndarray:
    return np.array([[1.0, 0.0], [0.0, -1.0]], dtype=np.complex128)

# -------------------- Model parameters --------------------
Delta = 1.0      # tunneling
eps = 0.0        # bias

H = 0.5 * eps * sigma_z() + 0.5 * Delta * sigma_x()
A = 0.5 * sigma_z()  # coupling operator (aka L)

# initial state |0><0|
rho0 = np.array([[1.0, 0.0], [0.0, 0.0]], dtype=np.complex128)

# -------------------- Bath parameters --------------------
temperature = 2.0   # T
omega_c = 10.0      # cutoff
alpha = 1.0         # overall coupling scale (shape parameter)

# Tabulate spectral density J(omega)
omega_max = 20.0
n_omega = 2048
omega = np.linspace(0.0, omega_max, n_omega, dtype=np.float64)
J = alpha * omega * np.exp(-omega / omega_c)

# -------------------- Simulation parameters --------------------
dt = 1e-2
n_steps = 1000
save_stride = 1
order = 4  # 0, 2, or 4

t_end = n_steps * dt
bcf_end_time = t_end

bath = taco.tcl.BathTabulated(
    temperature=temperature,
    omega=omega,
    J=J,
    bcf_end_time=bcf_end_time,
)
cfg = taco.tcl.SimConfig(
    dt=dt,
    n_steps=n_steps,
    save_stride=save_stride,
    order=order,
)

print("H=\n", H)
print("A=\n", A)
print("rho0=\n", rho0)
print("omega/J shapes:", omega.shape, J.shape)
print("t_end:", t_end)


### Run simulation (CPU)

This calls the existing C++ CPU TCL path and uses dense RK4 internally for propagation.

In [ ]:
res_cpu = taco.tcl.simulate(H, A, bath, cfg, rho0, device="cpu")
print(res_cpu.t.shape, res_cpu.rho.shape, res_cpu.rho.dtype)
print("trace(rho(t_end)) =", np.trace(res_cpu.rho[-1]))
print("rho(t_end) =\n", res_cpu.rho[-1])


### Optional: precompute BCF and reuse

If you sweep parameters but keep the same bath grid, precomputing `C(t_k)` can save time.
You can pass the result as `bcf=` into `simulate(...)` to skip the spectral-density → BCF FFT step.

In [ ]:
bcf = taco.tcl.precompute_bcf(bath, dt=dt)
print("bcf dtype/shape:", bcf.dtype, bcf.shape)

res_cpu_bcf = taco.tcl.simulate(H, A, bath, cfg, rho0, device="cpu", bcf=bcf)
print("max |rho_cpu - rho_cpu_bcf| =", np.max(np.abs(res_cpu.rho - res_cpu_bcf.rho)))


### Run simulation (CUDA)

If built with CUDA and a GPU is available, this uses the existing fused CUDA TCL4 builder and CUDA dense RK4.

- `precision="fp64"` (default) uses FP64 CUDA kernels
- `precision="fp32"` uses FP32 CUDA kernels (faster, more numerical error)


In [ ]:
res_gpu = None
if taco.build_info().get("cuda_enabled", False) and taco.cuda.is_available():
    res_gpu = taco.tcl.simulate(H, A, bath, cfg, rho0, device="cuda", precision="fp64", gpu_id=0)
    print("max |rho_cpu - rho_gpu| =", np.max(np.abs(res_cpu.rho - res_gpu.rho)))
else:
    print("CUDA not available in this build/runtime")


### Plot a simple observable (population)

If `matplotlib` is installed, plot the excited-state population `Re[rho_11(t)]`.

In [ ]:
try:
    import matplotlib.pyplot as plt

    pop1_cpu = res_cpu.rho[:, 1, 1].real
    plt.figure(figsize=(6, 3))
    plt.plot(res_cpu.t, pop1_cpu, label="CPU")
    if res_gpu is not None:
        pop1_gpu = res_gpu.rho[:, 1, 1].real
        plt.plot(res_gpu.t, pop1_gpu, '--', label="CUDA")
    plt.xlabel("t")
    plt.ylabel("Re[rho_11]")
    plt.legend()
    plt.tight_layout()
    plt.show()
except Exception as exc:
    print("Plot skipped:", exc)


In [2]:
# FP64 baseline
res64 = taco.tcl.e2e_cuda_compare_spin_boson(
    Nt_samples=200_000,
    dt=0.000625,
    temperature=2.0,
    omega_c=10.0,
    tidx="0:1:10000",
    threads=8,
    gpu_warmup=1,
    rk4_steps=0,
    precision="fp64",
    check=True,
)
res64


{'Nt_samples': 200000,
 'dt': 0.000625,
 'temperature': 2.0,
 'omega_c': 10.0,
 'precision': 'fp64',
 'cuda_enabled': True,
 'cuda_available': True,
 'tidx': [0,
  1,
  2,
  3,
  4,
  5,
  6,
  7,
  8,
  9,
  10,
  11,
  12,
  13,
  14,
  15,
  16,
  17,
  18,
  19,
  20,
  21,
  22,
  23,
  24,
  25,
  26,
  27,
  28,
  29,
  30,
  31,
  32,
  33,
  34,
  35,
  36,
  37,
  38,
  39,
  40,
  41,
  42,
  43,
  44,
  45,
  46,
  47,
  48,
  49,
  50,
  51,
  52,
  53,
  54,
  55,
  56,
  57,
  58,
  59,
  60,
  61,
  62,
  63,
  64,
  65,
  66,
  67,
  68,
  69,
  70,
  71,
  72,
  73,
  74,
  75,
  76,
  77,
  78,
  79,
  80,
  81,
  82,
  83,
  84,
  85,
  86,
  87,
  88,
  89,
  90,
  91,
  92,
  93,
  94,
  95,
  96,
  97,
  98,
  99,
  100,
  101,
  102,
  103,
  104,
  105,
  106,
  107,
  108,
  109,
  110,
  111,
  112,
  113,
  114,
  115,
  116,
  117,
  118,
  119,
  120,
  121,
  122,
  123,
  124,
  125,
  126,
  127,
  128,
  129,
  130,
  131,
  132,
  133,
  134,
  135,
 

In [3]:
# FP32 perf-only run (avoid raising on mismatches)
res32 = taco.tcl.e2e_cuda_compare_spin_boson(
    Nt_samples=200_000,
    dt=0.000625,
    temperature=2.0,
    omega_c=10.0,
    tidx="0:1:10000",
    threads=8,
    gpu_warmup=1,
    rk4_steps=0,
    precision="fp32",
    check=False,
)
res32


{'Nt_samples': 200000,
 'dt': 0.000625,
 'temperature': 2.0,
 'omega_c': 10.0,
 'precision': 'fp32',
 'cuda_enabled': True,
 'cuda_available': True,
 'tidx': [0,
  1,
  2,
  3,
  4,
  5,
  6,
  7,
  8,
  9,
  10,
  11,
  12,
  13,
  14,
  15,
  16,
  17,
  18,
  19,
  20,
  21,
  22,
  23,
  24,
  25,
  26,
  27,
  28,
  29,
  30,
  31,
  32,
  33,
  34,
  35,
  36,
  37,
  38,
  39,
  40,
  41,
  42,
  43,
  44,
  45,
  46,
  47,
  48,
  49,
  50,
  51,
  52,
  53,
  54,
  55,
  56,
  57,
  58,
  59,
  60,
  61,
  62,
  63,
  64,
  65,
  66,
  67,
  68,
  69,
  70,
  71,
  72,
  73,
  74,
  75,
  76,
  77,
  78,
  79,
  80,
  81,
  82,
  83,
  84,
  85,
  86,
  87,
  88,
  89,
  90,
  91,
  92,
  93,
  94,
  95,
  96,
  97,
  98,
  99,
  100,
  101,
  102,
  103,
  104,
  105,
  106,
  107,
  108,
  109,
  110,
  111,
  112,
  113,
  114,
  115,
  116,
  117,
  118,
  119,
  120,
  121,
  122,
  123,
  124,
  125,
  126,
  127,
  128,
  129,
  130,
  131,
  132,
  133,
  134,
  135,
 

In [4]:
def _gpu_total_ms(res: dict) -> float | None:
    l4 = res.get("l4", {})
    return l4.get("gpu_total_ms", None)

t64 = _gpu_total_ms(res64)
t32 = _gpu_total_ms(res32)
print("gpu_total_ms fp64:", t64)
print("gpu_total_ms fp32:", t32)
if t64 and t32:
    print("speedup (fp64/fp32):", t64 / t32)


gpu_total_ms fp64: 351.0962
gpu_total_ms fp32: 245.973
speedup (fp64/fp32): 1.4273769885312615
